# System Evaluation Plots
Generates all figures for the thesis report from per-bag JSON results and offline gesture performance data.

In [ ]:
import json
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
SCRIPT_DIR   = os.path.abspath('.')
BASE_DIR     = os.path.join(SCRIPT_DIR, '..')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')
GT_PATH      = os.path.join(BASE_DIR, 'ground_truth', 'ground_truth.json')
OFFLINE_PATH = os.path.join(BASE_DIR, '..', '..', 'offline_gesture_performance.txt')
FIGURES_DIR  = os.path.join(BASE_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

C = {
    'single':   '#2176AE',
    'multi':    '#E05C00',
    'total':    '#3DAA55',
    'correct':  '#3DAA55',
    'fp_auth':  '#E05C00',
    'no_auth':  '#BDBDBD',
    'match':    '#2176AE',
    'no_match': '#BDBDBD',
}

def savefig(name):
    path = os.path.join(FIGURES_DIR, name)
    plt.savefig(path + '.pdf')
    plt.savefig(path + '.png')
    print(f'Saved {name}')
    plt.show()

print('Setup complete.')

## 1. Load and Process Data

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _div(num, den):
    return num / float(den) if den > 0 else 0.0

def _f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def auth_metrics(data, gt_entry):
    gt_auth   = set(gt_entry.get('authorized_agents', []))
    pred_auth = set(data.get('authorized_agent_ids', []))
    all_ids   = set(data.get('all_agent_ids', [])) | gt_auth | pred_auth
    tp = len(pred_auth & gt_auth)
    fp = len(pred_auth - gt_auth)
    fn = len(gt_auth  - pred_auth)
    tn = len(all_ids  - pred_auth - gt_auth)
    prec = _div(tp, tp + fp)
    rec  = _div(tp, tp + fn)
    acc  = _div(tp + tn, tp + tn + fp + fn)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn,
                precision=prec, recall=rec, f1=_f1(prec, rec), accuracy=acc)

# ---------------------------------------------------------------------------
# Load ground truth
# ---------------------------------------------------------------------------
with open(GT_PATH) as f:
    ground_truth = json.load(f)

# ---------------------------------------------------------------------------
# Load per-bag JSONs and compute all metrics
# ---------------------------------------------------------------------------
records = []
for fpath in sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json'))):
    bag_name = os.path.splitext(os.path.basename(fpath))[0]
    if bag_name not in ground_truth:
        continue
    with open(fpath) as f:
        data = json.load(f)
    gt_entry      = ground_truth[bag_name]
    gt_auth_set   = set(gt_entry.get('authorized_agents', []))
    pred_auth_set = set(data.get('authorized_agent_ids', []))
    per_agent     = data.get('per_agent', {})

    am = auth_metrics(data, gt_entry)

    total_auth_time   = sum(v.get('total_time_authorized', 0.) for v in per_agent.values())
    correct_auth_time = sum(
        v.get('total_time_authorized', 0.)
        for k, v in per_agent.items() if int(k) in gt_auth_set
    )
    scene_dur = data.get('scene_duration', 0.)

    # Gesture ground truth
    gt_gestures    = gt_entry.get('expected_gestures', {})
    total_expected = sum(c for exp in gt_gestures.values() for c in exp.values()) if gt_gestures else 0
    total_detected = data.get('total_gestures', 0)
    auth_gestures  = data.get('authorized_gestures', 0)
    gt_auth_gest   = sum(
        v.get('gestures', 0)
        for k, v in per_agent.items() if int(k) in gt_auth_set
    )

    # Gesture TP/FP/FN
    gest_tp = gest_fp = gest_fn = 0
    for aid_str, expected in gt_gestures.items():
        detected = per_agent.get(str(aid_str), {}).get('gesture_counts', {})
        for gtype in set(expected) | set(detected):
            exp = expected.get(gtype, 0)
            det = detected.get(gtype, 0)
            gest_tp += min(exp, det)
            gest_fp += max(0, det - exp)
            gest_fn += max(0, exp - det)
    g_prec = _div(gest_tp, gest_tp + gest_fp)
    g_rec  = _div(gest_tp, gest_tp + gest_fn)

    lat      = data.get('gesture_latency', {})
    lat_mean = lat.get('mean', 0.)

    # TTA: GT agents that were authorized / FP agents that were authorized
    tta_gt_values = [
        v.get('time_to_auth')
        for k, v in per_agent.items()
        if int(k) in gt_auth_set and int(k) in pred_auth_set
        and v.get('time_to_auth') is not None
    ]
    tta_fp_values = [
        v.get('time_to_auth')
        for k, v in per_agent.items()
        if int(k) not in gt_auth_set and int(k) in pred_auth_set
        and v.get('time_to_auth') is not None
    ]

    # Scene confidence: GT agents vs all non-GT agents
    conf_gt_values = [
        v.get('avg_scene_confidence')
        for k, v in per_agent.items()
        if int(k) in gt_auth_set and v.get('avg_scene_confidence') is not None
    ]
    conf_non_gt_values = [
        v.get('avg_scene_confidence')
        for k, v in per_agent.items()
        if int(k) not in gt_auth_set and v.get('avg_scene_confidence') is not None
    ]

    records.append(dict(
        bag_name          = bag_name,
        scenario          = 'single' if bag_name.startswith('single_') else 'multi',
        scene_dur         = scene_dur,
        # auth
        auth_accuracy     = am['accuracy'],
        auth_precision    = am['precision'],
        auth_recall       = am['recall'],
        auth_f1           = am['f1'],
        auth_tp           = am['tp'], auth_fp=am['fp'], auth_fn=am['fn'], auth_tn=am['tn'],
        total_auth_time   = total_auth_time,
        correct_auth_time = correct_auth_time,
        fp_auth_time      = total_auth_time - correct_auth_time,
        no_auth_time      = max(0., scene_dur - total_auth_time),
        correct_auth_frac = _div(correct_auth_time, total_auth_time),
        # gestures
        gest_expected     = total_expected,
        gest_detected     = total_detected,
        gest_gt           = gt_auth_gest,
        gest_fp           = total_detected - gt_auth_gest,
        gest_tp           = gest_tp,
        gest_prec         = g_prec,
        gest_rec          = g_rec,
        gest_f1           = _f1(g_prec, g_rec),
        has_gest_gt       = bool(gt_gestures),
        lat_mean          = lat_mean,
        # new per-agent metrics
        tta_gt_values     = tta_gt_values,
        tta_fp_values     = tta_fp_values,
        conf_gt_values    = conf_gt_values,
        conf_non_gt_values= conf_non_gt_values,
    ))

single  = [r for r in records if r['scenario'] == 'single']
multi   = [r for r in records if r['scenario'] == 'multi']
print(f'Loaded {len(records)} bags  ({len(single)} single, {len(multi)} multi)')

In [ ]:
# ---------------------------------------------------------------------------
# Aggregate helper
# ---------------------------------------------------------------------------
def aggregate(recs):
    tp = sum(r['auth_tp'] for r in recs)
    fp = sum(r['auth_fp'] for r in recs)
    fn = sum(r['auth_fn'] for r in recs)
    tn = sum(r['auth_tn'] for r in recs)
    prec = _div(tp, tp + fp)
    rec  = _div(tp, tp + fn)
    acc  = _div(tp + tn, tp + tn + fp + fn)

    total_scene       = sum(r['scene_dur'] for r in recs)
    total_auth        = sum(r['total_auth_time'] for r in recs)
    correct_auth      = sum(r['correct_auth_time'] for r in recs)
    fp_auth           = sum(r['fp_auth_time'] for r in recs)
    no_auth           = sum(r['no_auth_time'] for r in recs)

    exp_total  = sum(r['gest_expected'] for r in recs)
    det_total  = sum(r['gest_detected'] for r in recs)
    gt_gest    = sum(r['gest_gt'] for r in recs)
    fp_gest    = sum(r['gest_fp'] for r in recs)
    g_tp_total = sum(r['gest_tp'] for r in recs)
    g_fp_total = sum(r['gest_fp'] for r in recs if r['has_gest_gt'])
    g_fn_total = exp_total - g_tp_total
    g_prec = _div(g_tp_total, g_tp_total + g_fp_total)
    g_rec  = _div(g_tp_total, g_tp_total + g_fn_total)

    lat_bags = [r['lat_mean'] for r in recs if r['lat_mean'] > 0]
    lat_mean = np.mean(lat_bags) if lat_bags else 0.

    tta_gt_all     = [v for r in recs for v in r['tta_gt_values']]
    tta_fp_all     = [v for r in recs for v in r['tta_fp_values']]
    conf_gt_all    = [v for r in recs for v in r['conf_gt_values']]
    conf_non_gt_all= [v for r in recs for v in r['conf_non_gt_values']]

    return dict(
        accuracy=acc, precision=prec, recall=rec, f1=_f1(prec, rec),
        total_scene=total_scene,
        correct_auth_frac  = _div(correct_auth, total_auth),
        fp_auth_frac       = _div(fp_auth, total_auth),
        no_auth_frac       = _div(no_auth, total_scene),
        correct_auth_time  = correct_auth,
        fp_auth_time       = fp_auth,
        no_auth_time       = no_auth,
        total_auth_time    = total_auth,
        gest_expected=exp_total, gest_detected=det_total,
        gest_gt=gt_gest, gest_fp_=fp_gest,
        gest_prec=g_prec, gest_rec=g_rec, gest_f1=_f1(g_prec, g_rec),
        lat_mean=lat_mean,
        tta_gt_all=tta_gt_all, tta_fp_all=tta_fp_all,
        conf_gt_all=conf_gt_all, conf_non_gt_all=conf_non_gt_all,
    )

agg_s = aggregate(single)
agg_m = aggregate(multi)
agg_t = aggregate(records)

for label, agg in [('Single', agg_s), ('Multi', agg_m), ('Total', agg_t)]:
    print(f"{label:6s}  Acc={agg['accuracy']:.3f}  P={agg['precision']:.3f}"
          f"  R={agg['recall']:.3f}  F1={agg['f1']:.3f}")

## Figure 1 — Offline Gesture Recognition Performance

In [ ]:
# Offline results (from offline_gesture_performance.txt)
offline = {
    'circle':      {'match': 145, 'no_match': 35,  'total': 180},
    'arm_up_down': {'match': 165, 'no_match': 28,  'total': 193},
    'OOD':         {'match': 120, 'no_match':  1,  'total': 121},
}
# OOD: 'match' = correctly rejected (TN), 'no_match' = falsely accepted (FP)

# Micro metrics (gesture classes only, excluding OOD)
offline_micro_prec = 310 / 311
offline_micro_rec  = 310 / 373
offline_micro_f1   = 2 * offline_micro_prec * offline_micro_rec / (offline_micro_prec + offline_micro_rec)
offline_ood_acc    = 120 / 121
offline_overall    = 430 / 494

classes    = list(offline.keys())
match_vals = [offline[c]['match']    for c in classes]
miss_vals  = [offline[c]['no_match'] for c in classes]
totals     = [offline[c]['total']    for c in classes]
accs       = [m / t for m, t in zip(match_vals, totals)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Left: stacked bar per class ---
ax = axes[0]
labels_disp = ['Circle', 'Arm Up-Down', 'OOD']
colors_bar  = [C['match'], C['match'], C['correct']]
miss_colors = [C['no_match'], C['no_match'], C['fp_auth']]

x = np.arange(len(classes))
bars_match = ax.bar(x, match_vals, color=[C['match'], C['match'], C['correct']],
                    label='Correct', zorder=3)
bars_miss  = ax.bar(x, miss_vals,  bottom=match_vals,
                    color=[C['no_match'], C['no_match'], C['fp_auth']],
                    label='Incorrect', zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(labels_disp)
ax.set_ylabel('Number of samples')
ax.set_title('(a) Samples per class')
ax.legend(loc='upper right')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

# Accuracy labels inside bars
for xi, (m, t) in enumerate(zip(match_vals, totals)):
    ax.text(xi, m / 2, f'{m/t:.1%}', ha='center', va='center',
            color='white', fontweight='bold', fontsize=10)

# --- Right: metrics bar ---
ax2 = axes[1]
metric_labels = ['Micro\nPrecision', 'Micro\nRecall', 'Micro\nF1', 'OOD\nAccuracy', 'Overall\nAccuracy']
metric_vals   = [offline_micro_prec, offline_micro_rec, offline_micro_f1,
                 offline_ood_acc, offline_overall]
bar_colors    = [C['single']] * 3 + [C['multi'], C['total']]

bars = ax2.bar(metric_labels, metric_vals, color=bar_colors, zorder=3)
ax2.set_ylim(0.7, 1.04)
ax2.set_ylabel('Score')
ax2.set_title('(b) Aggregate metrics')
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

for bar, val in zip(bars, metric_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.004,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Offline Gesture Recognition Performance (Leave-One-Out, N=11 participants)',
             fontsize=12, y=1.02)
plt.tight_layout()
savefig('fig_offline_gesture')

## Figure 2 — Online Authorization Metrics (Single vs Multi)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

metric_keys = ['accuracy', 'precision', 'recall', 'f1']
metric_disp = ['Accuracy', 'Precision', 'Recall', 'F1']
scenarios   = ['Single', 'Multi', 'Total']
agg_list    = [agg_s, agg_m, agg_t]
colors_bar  = [C['single'], C['multi'], C['total']]

n_metrics = len(metric_keys)
n_groups  = len(scenarios)
width     = 0.22
x         = np.arange(n_metrics)

for i, (label, agg, col) in enumerate(zip(scenarios, agg_list, colors_bar)):
    vals = [agg[k] for k in metric_keys]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=label, color=col, zorder=3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2.,
                bar.get_height() + 0.008,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metric_disp)
ax.set_ylim(0, 1.13)
ax.set_ylabel('Score')
ax.set_title('Online Authorization Performance — Single-Agent vs Multi-Agent')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
plt.tight_layout()
savefig('fig_auth_metrics')

## Figure 3 — Authorization Time Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

scenarios  = ['Single', 'Multi', 'Total']
agg_list   = [agg_s, agg_m, agg_t]

# Fractions of total scene time
correct_f = [a['correct_auth_time'] / a['total_scene'] for a in agg_list]
fp_f      = [a['fp_auth_time']      / a['total_scene'] for a in agg_list]
no_auth_f = [a['no_auth_time']      / a['total_scene'] for a in agg_list]

y = np.arange(len(scenarios))
ax.barh(y, correct_f, color=C['correct'],  label='Correct auth (GT agent)',  zorder=3)
ax.barh(y, fp_f,      left=correct_f,      color=C['fp_auth'], label='FP auth (wrong agent)', zorder=3)
ax.barh(y, no_auth_f, left=[c+f for c,f in zip(correct_f, fp_f)],
        color=C['no_auth'], label='No authorized agent', zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(scenarios)
ax.set_xlabel('Fraction of total scene time')
ax.set_xlim(0, 1.0)
ax.set_title('Authorization Time Breakdown')
ax.legend(loc='lower right', fontsize=9)
ax.xaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

# Annotate correct fraction
for yi, cf in enumerate(correct_f):
    if cf > 0.05:
        ax.text(cf / 2, yi, f'{cf:.1%}', ha='center', va='center',
                color='white', fontweight='bold', fontsize=9)

plt.tight_layout()
savefig('fig_auth_time_breakdown')

## Figure 4 — Per-Bag Authorization Metric Distributions

In [ ]:
def make_box_data(*lists):
    """Package unequal-length lists into a 1-D object array so that
    numpy (>=1.24) does not try to build a ragged 2-D array when
    matplotlib calls np.asarray() internally inside boxplot."""
    arr = np.empty(len(lists), dtype=object)
    for i, lst in enumerate(lists):
        arr[i] = list(lst)
    return arr

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric, title in [
    (axes[0], 'auth_f1',       'Authorization F1 per bag'),
    (axes[1], 'auth_accuracy', 'Authorization Accuracy per bag'),
]:
    data_s = [r[metric] for r in single]
    data_m = [r[metric] for r in multi]

    bp = ax.boxplot(
        make_box_data(data_s, data_m),
        labels=['Single', 'Multi'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linestyle='--'),
        flierprops=dict(marker='o', markersize=4, alpha=0.5),
        zorder=3,
    )
    for patch, col in zip(bp['boxes'], [C['single'], C['multi']]):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)

    # Overlay jitter
    for xi, (data, col) in enumerate(zip([data_s, data_m], [C['single'], C['multi']]), 1):
        jitter = np.random.uniform(-0.08, 0.08, len(data))
        ax.scatter(xi + jitter, data, color=col, alpha=0.5, s=18, zorder=4)

    ax.set_ylabel(metric.replace('auth_', '').title())
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.08)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

fig.suptitle('Per-Bag Authorization Performance Distribution', fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
savefig('fig_auth_distribution')

## Figure 5 — Gesture Performance (Expected vs Detected)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Left: stacked bar expected vs detected breakdown ---
ax = axes[0]
scenarios  = ['Single', 'Multi', 'Total']
agg_list   = [agg_s, agg_m, agg_t]
x = np.arange(len(scenarios))
width = 0.35

exp_vals  = [a['gest_expected'] for a in agg_list]
gt_vals   = [a['gest_gt']       for a in agg_list]
fp_vals   = [a['gest_fp_']      for a in agg_list]

ax.bar(x - width/2, exp_vals, width, color=C['no_auth'], label='Expected (GT)', zorder=3)
ax.bar(x + width/2, gt_vals,  width, color=C['correct'], label='Detected (GT agent)', zorder=3)
ax.bar(x + width/2, fp_vals,  width, bottom=gt_vals, color=C['fp_auth'],
       label='Detected (FP agent)', zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylabel('Number of gestures')
ax.set_title('(a) Expected vs. Detected gestures')
ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

# Recall annotation
for xi, (a, e, g) in enumerate(zip(agg_list, exp_vals, gt_vals)):
    if e > 0:
        ax.text(xi + width/2, g + fp_vals[xi] + 1.5,
                f"Recall={a['gest_rec']:.2f}",
                ha='center', fontsize=8, color='black')

# --- Right: Precision / Recall / F1 comparison ---
ax2 = axes[1]
metric_keys = ['gest_prec', 'gest_rec', 'gest_f1']
metric_disp = ['Precision', 'Recall', 'F1']
x2    = np.arange(len(metric_disp))
width2 = 0.25

for i, (label, agg, col) in enumerate(zip(scenarios, agg_list, [C['single'], C['multi'], C['total']])):
    vals = [agg[k] for k in metric_keys]
    offset = (i - 1) * width2
    bars = ax2.bar(x2 + offset, vals, width2, label=label, color=col, zorder=3)
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width() / 2.,
                 bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax2.set_xticks(x2)
ax2.set_xticklabels(metric_disp)
ax2.set_ylim(0, 1.2)
ax2.set_ylabel('Score')
ax2.set_title('(b) Gesture recognition metrics')
ax2.legend(fontsize=9)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

fig.suptitle('Online Gesture Performance', fontsize=12)
plt.tight_layout()
savefig('fig_gesture_performance')

## Figure 6 — Gesture Recognition Latency Distribution

In [ ]:
lat_single = [r['lat_mean'] for r in single if r['lat_mean'] > 0]
lat_multi  = [r['lat_mean'] for r in multi  if r['lat_mean'] > 0]

fig, ax = plt.subplots(figsize=(8, 4))

bins = np.linspace(0, 6, 20)
ax.hist(lat_single, bins=bins, color=C['single'], alpha=0.7, label=f'Single (n={len(lat_single)})', zorder=3)
ax.hist(lat_multi,  bins=bins, color=C['multi'],  alpha=0.7, label=f'Multi  (n={len(lat_multi)})',  zorder=3)

ax.axvline(np.mean(lat_single), color=C['single'], linestyle='--', linewidth=1.5,
           label=f'Single mean={np.mean(lat_single):.2f}s')
ax.axvline(np.mean(lat_multi),  color=C['multi'],  linestyle='--', linewidth=1.5,
           label=f'Multi mean={np.mean(lat_multi):.2f}s')

ax.set_xlabel('Mean recognition latency per bag (s)')
ax.set_ylabel('Number of bags')
ax.set_title('Gesture Recognition Latency Distribution')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
plt.tight_layout()
savefig('fig_latency_distribution')

## Figure 7 — Time to Authorization Distribution

In [ ]:
tta_single_gt = [v for r in single for v in r['tta_gt_values']]
tta_multi_gt  = [v for r in multi  for v in r['tta_gt_values']]

fig, ax = plt.subplots(figsize=(8, 4))

bins = np.linspace(0, 12, 25)
ax.hist(tta_single_gt, bins=bins, color=C['single'], alpha=0.7,
        label=f'Single GT (n={len(tta_single_gt)})', zorder=3)
ax.hist(tta_multi_gt,  bins=bins, color=C['multi'],  alpha=0.7,
        label=f'Multi GT  (n={len(tta_multi_gt)})',  zorder=3)

if tta_single_gt:
    ax.axvline(np.mean(tta_single_gt), color=C['single'], linestyle='--', linewidth=1.5,
               label=f'Single mean={np.mean(tta_single_gt):.2f}s')
if tta_multi_gt:
    ax.axvline(np.mean(tta_multi_gt),  color=C['multi'],  linestyle='--', linewidth=1.5,
               label=f'Multi mean={np.mean(tta_multi_gt):.2f}s')

ax.set_xlabel('Time to first authorization — GT agents (s)')
ax.set_ylabel('Number of agents')
ax.set_title('Time to Authorization Distribution (Ground-Truth Agents)')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
plt.tight_layout()
savefig('fig_time_to_auth')

## Figure 8 — Per-Bag Gesture Recall (Sorted)

In [ ]:
# Only bags that have GT gesture labels
recs_with_gt = [r for r in records if r['has_gest_gt']]
recs_single_gt = sorted([r for r in recs_with_gt if r['scenario'] == 'single'],
                         key=lambda r: r['gest_rec'])
recs_multi_gt  = sorted([r for r in recs_with_gt if r['scenario'] == 'multi'],
                         key=lambda r: r['gest_rec'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, recs, col, title in [
    (axes[0], recs_single_gt, C['single'], 'Single-Agent'),
    (axes[1], recs_multi_gt,  C['multi'],  'Multi-Agent'),
]:
    names  = [r['bag_name'].replace('single_', '').replace('multi_', '') for r in recs]
    recalls = [r['gest_rec'] for r in recs]
    ax.barh(names, recalls, color=col, alpha=0.8, zorder=3)
    ax.axvline(np.mean(recalls), color='black', linestyle='--', linewidth=1.2,
               label=f'Mean={np.mean(recalls):.2f}')
    ax.set_xlim(0, 1.1)
    ax.set_xlabel('Gesture Recall')
    ax.set_title(f'({"a" if col == C["single"] else "b"}) {title}')
    ax.legend(fontsize=9)
    ax.xaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

axes[0].set_ylabel('Bag index')
fig.suptitle('Per-Bag Gesture Recall (sorted)', fontsize=12)
plt.tight_layout()
savefig('fig_per_bag_gesture_recall')

## Figure 9 — Scene Confidence: GT vs Non-GT Agents

In [ ]:
# Collect confidence values per scenario
conf_gt_single    = [v for r in single for v in r['conf_gt_values']]
conf_non_gt_single= [v for r in single for v in r['conf_non_gt_values']]
conf_gt_multi     = [v for r in multi  for v in r['conf_gt_values']]
conf_non_gt_multi = [v for r in multi  for v in r['conf_non_gt_values']]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)

# --- Left: boxplot with jitter ---
ax = axes[0]
groups = [conf_gt_single, conf_non_gt_single, conf_gt_multi, conf_non_gt_multi]
labels = ['GT\n(single)', 'Non-GT\n(single)', 'GT\n(multi)', 'Non-GT\n(multi)']
colors = [C['single'], C['no_auth'], C['multi'], C['no_auth']]

bp = ax.boxplot(
    make_box_data(*groups),
    labels=labels, patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linestyle='--'),
    flierprops=dict(marker='o', markersize=4, alpha=0.4),
    zorder=3)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.65)

for xi, (grp, col) in enumerate(zip(groups, colors), 1):
    jitter = np.random.uniform(-0.12, 0.12, len(grp))
    ax.scatter(xi + jitter, grp, color=col, alpha=0.45, s=14, zorder=4)

for xi, grp in enumerate(groups, 1):
    if grp:
        ax.text(xi, np.mean(grp) + 0.015, f'{np.mean(grp):.3f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylabel('Average scene confidence')
ax.set_title('(a) Confidence distribution per group')
ax.set_ylim(0, 1.08)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

# --- Right: mean bar comparison ---
ax2 = axes[1]
group_labels = ['Single\nGT', 'Single\nNon-GT', 'Multi\nGT', 'Multi\nNon-GT']
means = [np.mean(g) if g else 0 for g in groups]
bar_colors = [C['single'], C['no_auth'], C['multi'], C['no_auth']]

bars = ax2.bar(group_labels, means, color=bar_colors, zorder=3, alpha=0.85)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Mean scene confidence')
ax2.set_title('(b) Mean confidence by agent type')
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

gt_patch  = mpatches.Patch(color=C['single'], alpha=0.7, label='GT agent (single)')
ngt_patch = mpatches.Patch(color=C['no_auth'], alpha=0.7, label='Non-GT agent')
mgt_patch = mpatches.Patch(color=C['multi'],  alpha=0.7, label='GT agent (multi)')
ax2.legend(handles=[gt_patch, mgt_patch, ngt_patch], fontsize=9, loc='lower right')

fig.suptitle('Scene Confidence: GT vs Non-GT Agents', fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
savefig('fig_confidence_separation')

## Figure 10 — TTA: GT vs FP Agents (Multi-Agent)

In [ ]:
tta_multi_gt = [v for r in multi for v in r['tta_gt_values']]
tta_multi_fp = [v for r in multi for v in r['tta_fp_values']]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Left: overlapping histograms ---
ax = axes[0]
bins = np.linspace(0, 14, 22)
ax.hist(tta_multi_gt, bins=bins, color=C['correct'],  alpha=0.7,
        label=f'GT agents  (n={len(tta_multi_gt)})', zorder=3)
ax.hist(tta_multi_fp, bins=bins, color=C['fp_auth'], alpha=0.7,
        label=f'FP agents  (n={len(tta_multi_fp)})', zorder=3)
if tta_multi_gt:
    ax.axvline(np.mean(tta_multi_gt), color=C['correct'],  linestyle='--', linewidth=1.5,
               label=f'GT mean={np.mean(tta_multi_gt):.2f}s')
if tta_multi_fp:
    ax.axvline(np.mean(tta_multi_fp), color=C['fp_auth'], linestyle='--', linewidth=1.5,
               label=f'FP mean={np.mean(tta_multi_fp):.2f}s')
ax.set_xlabel('Time to first authorization (s)')
ax.set_ylabel('Number of agents')
ax.set_title('(a) TTA distribution — multi-agent')
ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

# --- Right: boxplot comparison GT vs FP ---
ax2 = axes[1]
groups = [tta_multi_gt, tta_multi_fp]
bp = ax2.boxplot(
    make_box_data(*groups),
    labels=['GT agents', 'FP agents'], patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linestyle='--'),
    flierprops=dict(marker='o', markersize=5, alpha=0.5),
    zorder=3)
box_colors = [C['correct'], C['fp_auth']]
for patch, col in zip(bp['boxes'], box_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.65)

for xi, (grp, col) in enumerate(zip(groups, box_colors), 1):
    jitter = np.random.uniform(-0.1, 0.1, len(grp))
    ax2.scatter(xi + jitter, grp, color=col, alpha=0.55, s=20, zorder=4)
    if grp:
        ax2.text(xi, np.mean(grp) + 0.2, f'mean={np.mean(grp):.2f}s',
                 ha='center', fontsize=9, fontweight='bold')

ax2.set_ylabel('Time to first authorization (s)')
ax2.set_title('(b) TTA boxplot — GT vs FP')
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)

fig.suptitle('Time to Authorization: GT vs FP Agents (Multi-Agent Scenarios)', fontsize=12)
plt.tight_layout()
savefig('fig_tta_gt_vs_fp')

## Figure 11 — Per-Bag Correct Authorization Time (%)

In [ ]:
# Sort all bags by correct_auth_frac, color by scenario
all_sorted = sorted(records, key=lambda r: r['correct_auth_frac'])
names      = [r['bag_name'].replace('single_', 'S-').replace('multi_', 'M-') for r in all_sorted]
fracs      = [r['correct_auth_frac'] * 100 for r in all_sorted]
colors_bar = [C['single'] if r['scenario'] == 'single' else C['multi'] for r in all_sorted]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(names, fracs, color=colors_bar, alpha=0.85, zorder=3)

# Aggregate reference lines
agg_single_frac = agg_s['correct_auth_frac'] * 100
agg_multi_frac  = agg_m['correct_auth_frac'] * 100
agg_total_frac  = agg_t['correct_auth_frac'] * 100

ax.axvline(agg_single_frac, color=C['single'], linestyle='--', linewidth=1.4,
           label=f'Single agg. {agg_single_frac:.1f}%')
ax.axvline(agg_multi_frac,  color=C['multi'],  linestyle='--', linewidth=1.4,
           label=f'Multi agg.  {agg_multi_frac:.1f}%')
ax.axvline(agg_total_frac,  color=C['total'],  linestyle=':',  linewidth=1.4,
           label=f'Total agg.  {agg_total_frac:.1f}%')

ax.set_xlabel('Correct authorization time (%)')
ax.set_xlim(0, 110)
ax.set_title('Per-Bag Correct Authorization Time (sorted)\nBlue = single-agent  |  Orange = multi-agent')
ax.legend(fontsize=9, loc='lower right')
ax.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)

# Patches for legend
s_patch = mpatches.Patch(color=C['single'], alpha=0.8, label='Single-agent')
m_patch = mpatches.Patch(color=C['multi'],  alpha=0.8, label='Multi-agent')
ax.legend(handles=[s_patch, m_patch,
                   plt.Line2D([0],[0], color=C['single'], linestyle='--', label=f'Single agg. {agg_single_frac:.1f}%'),
                   plt.Line2D([0],[0], color=C['multi'],  linestyle='--', label=f'Multi agg.  {agg_multi_frac:.1f}%'),
                   plt.Line2D([0],[0], color=C['total'],  linestyle=':',  label=f'Total agg.  {agg_total_frac:.1f}%'),
                  ], fontsize=8, loc='lower right')

plt.tight_layout()
savefig('fig_correct_auth_time_per_bag')

## Summary

In [ ]:
print('Figures saved to:', FIGURES_DIR)
print()
print('Files generated:')
for f in sorted(os.listdir(FIGURES_DIR)):
    print(' ', f)